# Blazar Population ML — Project 1

**Goal:** Apply classical ML to the 4LAC-DR3 blazar catalog to recover constraints on jet baryon loading ($\mu$) from multi-wavelength observational features, and to characterize the blazar population via unsupervised clustering.

**Reference:** Rueda-Becerril, Harrison & Giannios (2021, MNRAS 501, 4092)

**Catalogs:**
- 4LAC-DR3: Ajello et al. (2022, arXiv:2209.12070), high-latitude, $|b| > 10^\circ$, `table-4LAC-DR3-h.fits`
- MOJAVE-XVII: apparent velocities from VLBI monitoring, `VizieR-MOJAVE-XVII.fit`

---

## Analysis Structure

- **Part 1 (full catalog, ~3,400 sources):** catalog-native features only, no kinematics
- **Part 2 (MOJAVE cross-matched, ~334 sources):** adds $\beta_{\text{app}}$ and $\Gamma_{\text{min}}$

**Notebook sections:**
1. Setup and imports
2. Load and inspect 4LAC-DR3
3. Load and inspect MOJAVE-XVII
4. Cross-match: 4LAC-DR3 × MOJAVE-XVII
5. Feature derivation and unified dataframes
6. Missingness audit
7. Exploratory data analysis (EDA)
8. Dimensionality reduction: PCA and UMAP
9. Clustering: GMM and HDBSCAN

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from astropy.io import fits
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings('ignore')

# Cosmology (consistent with 2021 paper)
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# Paths
DATA_DIR = Path('data/')
LAC_H  = DATA_DIR / 'table-4LAC-DR3-h.fits'
MOJAVE = DATA_DIR / 'VizieR-MOJAVE-XVII.fit'

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
CLASS_COLORS = {'bll': 'steelblue', 'fsrq': 'tomato', 'bcu': 'gray'}
SED_COLORS   = {'lsp': 'tomato', 'isp': 'goldenrod', 'hsp': 'steelblue'}

: 

## 2. Load and Inspect 4LAC-DR3

Use `table-4LAC-DR3-h.fits` (high Galactic latitude, $|b| > 10^\circ$) as the primary catalog.
`table-4LAC-DR3-l.fits` covers low-latitude sources and is excluded from the main analysis.

**Tasks:**
- [X] Print header data unit (HDU) structure and column list with units
- [X] Check row count: expect 3,407 sources
- [X] Print CLASS and SED_class distributions (note mixed-case entries: normalize to lowercase)
- [X] Check for masked values vs. sentinel zeros, especially in `Redshift`, `nu_syn`, `nuFnu_syn`

### Inspecting `table-4LAC-DR3-h.fits`

In [ ]:
def safe_float(col):
    return np.ma.filled(col.data, np.nan).astype(float)

def safe_str(col):
    return np.array([str(v).strip() if v is not np.ma.masked else '' for v in col])

def safe_str_lower(col):
    return np.array([str(v).strip().lower() if v is not np.ma.masked else '' for v in col])

In [ ]:
# Exploring header data unit (HDU)
with fits.open(LAC_H) as hdul:
    for i, hdu in enumerate(hdul):
        print(f"HDU {i}: {type(hdu).__name__}")
        if hasattr(hdu, 'columns') and hdu.columns:
            print(f"Rows: {len(hdu.data)}")
            print(f"Columns ({len(hdu.columns)}):")
            for col in hdu.columns:
                print(f"  {col.name:30s}  {col.format:6s}  {col.unit if col.unit else ''}")

In [ ]:
lac = Table.read(LAC_H)
lac_len = len(lac)

cls_mixed = safe_str(lac['CLASS'])
cls_lower = safe_str_lower(lac['CLASS'])
sed_arr = safe_str_lower(lac['SED_class'])

# Finding unique values in CLASS column
classes, cls_counts = np.unique(cls_lower, return_counts=True)
classes_mixed, counts_mixed = np.unique(cls_mixed, return_counts=True)

# Finding unique values in SED_class column
sed, sed_counts = np.unique(sed_arr, return_counts=True)


### `CLASS` distribution

In [ ]:
for cl, n in sorted(zip(classes_mixed, counts_mixed), key=lambda x: -x[1]):
    print(f"{cl:10s}  {n:5d}  ({100 * n / lac_len:.1f}%)")

### `SED_class` distribution

In [ ]:
for s, n in sorted(zip(sed, sed_counts), key=lambda x: -x[1]):
    label = s if s else '(missing)'
    print(f"{label:10s}  {n:5d}  ({100 * n / lac_len:.1f}%)")

### Missingness for key features

In [ ]:
key_cols = [
    'PL_Index', 'LP_Index', 'Redshift', 'nu_syn', 'nuFnu_syn', 'HE_EPeak',
    'HE_nuFnuPeak', 'Variability_Index', 'Frac_Variability', 'Energy_Flux100'
]

for col in key_cols:
    arr = safe_float(lac[col])
    n_nan = np.sum(np.isnan(arr))
    n_zero = np.sum(arr == 0)
    print(f"{col:25s}  missing: {n_nan:4d} ({100 * n_nan / lac_len:5.1f}%)  zero: {n_zero:4d}")
    
print(f"\nN = {lac_len}")

### Derived features feasibility

In [ ]:
he = safe_float(lac['HE_nuFnuPeak'])
syn = safe_float(lac['nuFnu_syn'])
flux = safe_float(lac['Energy_Flux100'])
z = np.nan_to_num(lac['Redshift'].data, nan=0.0, posinf=0.0, neginf=0.0)

has_sed = (syn > 0) & (he > 0) & (~np.isnan(he))
has_z = z > 0
has_lum = has_z & (flux > 0) & (~np.isnan(flux))
has_cd = has_sed
is_blazar = np.isin(cls_lower, ['bll', 'fsrq'])
is_bcu = cls_lower == 'bcu'

### Compton Dominand and Gamma-ray Luminosity

In [ ]:
cd_valid = np.sum(has_cd)
lum_valid = np.sum(has_lum)

print(f"Compton dominance computable:    {cd_valid:4d}/{lac_len}  ({100 * cd_valid / lac_len:.1f}%)")
print(f"Gamma-ray luminosity computable: {lum_valid:4d}/{lac_len}  ({100 * lum_valid / lac_len:.1f}%)")

### Redshift

Redshift 0 means "no measurement", not actually z = 0.

In [ ]:
print(f"z > 0 (measured):    {np.sum(z > 0):4d} ({100 * np.sum(z > 0) / lac_len:.1f}%)")
print(f"z = 0 (no redshift): {np.sum(z == 0):4d} ({100 * np.sum(z == 0) / lac_len:.1f}%)")

### `CLASS` case normalization issue

Mixed case entries detected: bll vs BLL, fsrq vs FSRQ; need normalization

In [ ]:
for cl, n in sorted(zip(classes, cls_counts), key=lambda x: -x[1]):
    print(f"{cl:5s} + {cl.upper():5s} = {n:5d} ({100 * n / cls_counts.sum():.1f}%)")

### BCU SED class breakdown

In [ ]:
bcu_sed, bcu_counts = np.unique(sed_arr[is_bcu], return_counts=True)

for s, n in sorted(zip(bcu_sed, bcu_counts), key=lambda x: -x[1]):
    label = s if s else '(no SED class)'
    print(f"{label:15s}  {n:4d}")

### Confirmed blazars (BL Lacs + FSRQs) by SED class

In [ ]:
bl_sed, bl_counts = np.unique(sed_arr[is_blazar], return_counts=True)

for s, n in sorted(zip(bl_sed, bl_counts), key=lambda x: -x[1]):
    label = s if s else '(no SED class)'
    print(f"{label:15s}  {n:4d}")

### Summary

**Effective Sample Sizes**

In [ ]:
print(f"Total sources:                     {lac_len:4d}")
print(f"Confirmed blazars (BL Lac + FSRQ): {np.sum(is_blazar):4d}  ({100 * np.sum(is_blazar) / lac_len:.1f}%)")
print(f"BCU:                               {np.sum(is_bcu):4d}  ({100 * np.sum(is_bcu) / lac_len:.1f}%)")
print(f"With SED peak (nu_syn > 0):        {np.sum(syn > 0):4d}  ({100 * np.sum(syn > 0) / lac_len:.1f}%)")
print(f"With Compton dominance:            {np.sum(has_cd):4d}  ({100 * np.sum(has_cd) / lac_len:.1f}%)")
print(f"With redshift:                     {np.sum(has_z):4d}  ({100 * np.sum(has_z) / lac_len:.1f}%)")
print(f"With luminosity (z * flux):        {np.sum(has_lum):4d}  ({100 * np.sum(has_lum) / lac_len:.1f}%)")
print(f"Confirmed blazars + all above:     {np.sum(is_blazar & has_cd & has_lum):4d}  ({100 * np.sum(is_blazar & has_cd & has_lum) / lac_len:.1f}%)")

**1. `CLASS` distribution (after normalization):**
- BL Lac: 1,379 (40.5%)
- BCU: 1,208 (35.5%) -- largest uncertainty population
- FSRQ: 755 (22.2%)
- Radio Galaxy: 42 (1.2%)
- Other (NLSY1, AGN, CSS, SSRQ, SEY): 23 (~0.6%)

**2. `SED_class` distribution:**
- Shows Low synchrotron peaked (LSP) sources dominate at 45.9%, while 777 entries (22.8%) missing SED classification.
- The missing SED class correspond exactly to sources lacking synchrotron peak measurements (i.e., zero `nu_syn` or `nuFnu_syn` values), predominantly BCUs.
- High synchrotron peaked (HSP) and Intermediate synchrotron peaked (ISP) make up the remaining 31.3% of classified sources.

**3.Missingness:**
- Looking at the missing patterns, `PL_index` and `LP_index` are fully populated, but there are gaps elsewhere that need investigation.
- Redshift is encoded as 0 for unmeasured sources rather than NaN.
- `HE_EPeak` has 518 missing entries (15.2%), while `Energy_Flux100` is complete.
- For derived features, Compton dominance has good coverage at 66%, but gamma-ray luminosity is constrained by redshift availability at only 53%.

Looking at the effective sample sizes, the full catalog has 3,407 sources, but only 1,806 have redshift data needed for luminosity calculations, and just 1,208 are BCUs (the actual classification targets).

The confirmed blazars (BLL + FSRQ) total 2,134 sources, which gives a solid training set.

---

## 3. Load and Inspect MOJAVE-XVII

The file has 6 HDUs. The relevant ones are:
- **HDU 1 (204 sources):** BL Lac-dominated. Columns include `ID`, `_RA`, `_DE`, `betaMax`, `e_betaMax`, `Cl`
- **HDU 2 (230 sources):** FSRQ/Quasar-dominated. Same key columns, also has `Smax` (peak flux)
- **HDU 4 (1,743 rows):** per-component kinematics — one row per tracked jet component, not used here

HDU 1 and HDU 2 have **zero overlap** in source IDs. Combine and deduplicate to get 434 unique sources.

**Warning:** The `SimbadName` column in HDUs 1 and 2 has a malformed FITS format that will crash `Table(hdul[i].data)`. Extract only the columns you need by iterating over `hdul[i].data` rows directly.

**Tasks:**
- [ ] Load HDU 1 and HDU 2, extract: `ID`, `_RA`, `_DE`, `betaMax`, `e_betaMax`, `Cl`
- [ ] Confirm zero overlap between HDU 1 and HDU 2 source IDs
- [ ] Combine and deduplicate on `ID`
- [ ] Report `betaMax` distribution; how many sources have betaMax = 0?

### Inspecting HDU content

In [ ]:
with fits.open(MOJAVE) as hdul:
    print(f"HDUs: {len(hdul)}")
    for i, hdu in enumerate(hdul):
        print(f"\nHDU {i}: {type(hdu).__name__}")
        if hasattr(hdu, 'columns') and hdu.columns:
            print(f"  Rows: {len(hdu.data)}")
            print(f"  Columns ({len(hdu.columns)}):")
            for col in hdu.columns:
                print(f"    {col.name:30s}  {col.format:6s}  {col.unit if col.unit else ''}")

`VizieR-MOJAVE-XVII.fit` file has 6 HDUs:

- **HDU 1**: 204 rows, source list with max apparent velocity `betaMax`
    - this appears to be the main source table
- **HDU 2**: 230 rows, another source list with similar structure but slightly different columns
    - includes `Smax`, the peak flux density
- **HDU 3**: 39,294 rows, individual component measurements
    - VLBI component data
- **HDU 4**: 1,743 rows, individual component kinematics
    - per-component proper motions, beta
- **HDU 5**: 881 rows, component kinematics with acceleration data

For the apparent velocity cross-match we need to focus on the key columns: `betaMax` and its uncertainty `e_betaMax` from HDUs 1 and 2, along with source identifiers (in B1950 format), coordinates for matching, and redshift values.

HDU 4 contains individual component apparent velocities with their uncertainties. The tricky part is that HDU 1 has 204 sources while HDU 2 has 230 sources.

### Inspecting data distributions

In [ ]:
with fits.open(MOJAVE) as hdul:
    # Check HDU 1 and 2 (the source level tables)
    for hdu_idx in [1, 2]:
        hdu = hdul[hdu_idx]
        data = hdu.data
        print(f"=== HDU {hdu_idx} ({len(data)} sources) ===")
        
        # Sample IDs and coordinates
        ids = [str(r['ID']).strip() for r in data[:5]]
        ras = [r['_RA'] for r in data[:5]]
        decs = [r['_DE'] for r in data[:5]]
        betas = [r['betaMax'] for r in data[:5]]
        print(f"  Sample IDs:      {ids}")
        print(f"  Sample RAs:      {[f'{r:.4f}' for r in ras]}")
        print(f"  Sample DECs:     {[f'{d:.4f}' for d in decs]}")
        print(f"  Sample betaMax:  {[f'{b:.4f}' for b in betas]}")
        
        # betaMax distribution
        betas_all = np.array([r['betaMax'] for r in data])
        print(f"  betaMax range: {betas_all.min():.2f} - {betas_all.max():.2f}")
        print(f"  betaMax median: {np.median(betas_all):.2f}")

        # Redshift coverage
        zs = np.array([r['z'] for r in data])
        print(f"  z > 0: {np.sum(zs > 0)} / {len(data)}")

        # Class column
        cls = [str(r['Cl']).strip() for r in data]
        unique_cls, counts = np.unique(cls, return_counts=True)
        print(f"  Classes: {dict(zip(unique_cls, counts))}")

    # HDU 4: per-component kinematics, checking if useful for individual beta
    hdu4 = hdul[4].data
    print(f"=== HDU 4 sample (per-component beta) ===")
    unique_ids = list(set([str(r['ID']).strip() for r in hdu4]))
    print(f"  Unique sources in HDU4: {len(unique_ids)}")
    betas4 = np.array([r['beta'] for r in hdu4])
    print(f"  beta range: {betas4.min():.3f} - {betas4.max():.3f}")

From this inspections we find:

- **HDU 1**: 204 sources that appear to be:
    - BL Lacs (79 B-class),
    - Quasars (101 Q-class),
    - Galaxies (16 G-class),
    - Narrow-line (4 N-class),
    - Unknown (4 U-class).

    IDs are in B1950 format. betaMax median = 1.44 (lower)

- **HDU 2**: 230 sources:
    - 175 Q-class strongly Quasar-dominated,
    - 38 B-class BL Lacs.
    
    betaMax median = 7.02 (higher, as expected for quasars). Different sources from HDU 1.

- **HDU 4**: 382 unique sources with per-component kinematics

This MOJAVE XVII catalog appears to split between two distinct populations:
- HDU 1 likely represents the BL Lac sample from an earlier MOJAVE selection, while
- HDU 2 contains the flux-density limited or kinematics-selected sample with more quasars.
    
The "XVII" designation points to the MOJAVE XVII paper on parsec-scale jet kinematics, with each HDU corresponding to different selection criteria used in the program.

### Combining MOJAVE HUDs

In [ ]:
def load_mojave_hdu(hdul, hdu_idx):
    data = hdul[hdu_idx].data
    return Table({
        'ID': safe_str(data['ID']),
        'RA': safe_float(data['_RA']),
        'DEC': safe_float(data['_DE']),
        'betaMax': safe_float(data['betaMax']),
        'e_betaMax': safe_float(data['e_betaMax']),
        'Cl': safe_float(data['e_betaMax'])
    })

In [ ]:
with fits.open(MOJAVE) as hdul:
    moj_combined = vstack([load_mojave_hdu(hdul, 1), load_mojave_hdu(hdul, 2)])

_, first_idx = np.unique(moj_combined['ID'], return_index=True)
mojave = moj_combined[np.sort(first_idx)]
mojave

### Selectinv valid sources

In [ ]:
lac_ra = safe_float(lac['RA_Counterpart'])
lac_dec = safe_float(lac['DEC_Counterpart'])
valid = (~np.isnan(lac_ra)) & (~np.isnan(lac_dec))
lac_sub = lac[valid]

lac_coords = SkyCoord(ra=lac_ra[valid] * u.deg, dec=lac_dec[valid] * u.deg)
moj_coords = SkyCoord(ra=mojave['RA'] * u.deg, dec=mojave['DEC'] * u.deg)
idx, sep, _ = moj_coords.match_to_catalog_sky(lac_coords)
sep_arcsec = sep.to(u.arcsec).value

print("=== Tolerance sensitivity ===")
for tol in [1.0, 2.0, 3.0, 5.0]:
    print(f"  {tol}\": {np.sum(sep_arcsec < tol)} matched")

In [ ]:
# Selecting sources within tolerance threshold
tol = 2.0
mask = sep_arcsec < tol
matched_moj = mojave[mask]
matched_lac = lac_sub[idx[mask]]
seps = sep_arcsec[mask]

# Cross-match summary
print(f"Matched: {len(matched_moj)} / {len(mojave)} MOJAVE sources")
print(f"Sep: median = {np.median(seps):.4f}\"  max = {seps.max():.4f}\"")

In [ ]:
# CLASS
cls_match = safe_str_lower(matched_lac['CLASS'])
for c, n in sorted(zip(*np.unique(cls, return_counts=True)), key=lambda x: -x[1]):
    print(f"{c:8s}: {n}")

In [ ]:
# SED_class
sed_match = safe_str_lower(matched_lac['CLASS'])
for s, n in sorted(zip(*np.unique(sed, return_counts=True)), key=lambda x: -x[1]):
    print(f"{(s if s else '(none)'):8s}: {n}")

## 4. Cross-Match: 4LAC-DR3 × MOJAVE-XVII

Use `SkyCoord.match_to_catalog_sky` for positional matching.

- **4LAC-DR3 coords:** `RA_Counterpart` / `DEC_Counterpart` (counterpart position, more precise than LAT position). Filter out sources with masked counterpart coords before building the SkyCoord.
- **MOJAVE coords:** `_RA`, `_DE` (J2000, degrees)

Test tolerances 1", 2", 3", 5". If match count is stable across this range, 2" is safe — VLBI positions are precise enough that all real matches will be well under 1".

**Tasks:**
- Run cross-match, report match count and separation statistics
- Inspect CLASS and SED_class breakdown of matched sources
- Note the selection bias: MOJAVE selects radio-bright sources, mostly LSP/FSRQ

### Loading 4LAC-DR3


Let's verify whether these are overlapping or distinct samples. Since the source IDs might not align directly with 4LAC-DR3, I'll need to rely on coordinate-based cross-matching. HDU 2 also includes peak flux density (Smax), which HDU 1 lacks, so I need to decide which table to prioritize for the analysis.

For the cross-match itself, I'll use positional matching with a ~1-2 arcsecond tolerance, comparing MOJAVE's _RA and _DE against 4LAC-DR3's RA_Counterpart and DEC_Counterpart. Let me examine the actual source populations in both HDUs to determine which one to use.

For cross-matching, I'm looking at 204 plus 230 sources across the two HDUs, totaling around 434 with potential overlap to verify. The betaMax values in HDU 1 range from zero to 34.8 with a median of 1.44, where zeros likely indicate unresolved sources or missing superluminal motion measurements. My strategy is to use positional matching with the 4LAC-DR3 coordinates along with the VLBI counterpart names, applying a tight angular tolerance under 2 arcseconds to ensure reliable associations.

Now I'm proceeding with the actual cross-match and building the unified dataframe. I noticed the roadmap references "MOJAVE DR3" but the file is actually MOJAVE XVII from a specific data release paper—I should update that for clarity. The zero betaMax entries represent sources without detected proper motion, which is physically valid since some jets don't exhibit superluminal motion at 15 GHz cadence, though they could also indicate single-epoch observations. I'll keep these entries but mark them appropriately. The key takeaway is that MOJAVE XVII contains two sub-tables with 204 and 230 sources respectively.

In [ ]:
# Cross-match 4LAC-DR3 x MOJAVE-XVII


## 5. Feature Derivation and Unified Dataframes

Build two clean pandas DataFrames.

**Part 1 — all 3,407 sources, catalog-native + derived features:**

| Feature | Source column | Notes |
|---------|--------------|-------|
| `class` | `CLASS` | Lowercase, normalize |
| `sed_class` | `SED_class` | Lowercase |
| `redshift` | `Redshift` | Treat 0 as NaN |
| `pl_index` | `PL_Index` | Photon spectral index |
| `lp_alpha` | `LP_Index` | Log-parabola α |
| `lp_beta` | `LP_beta` | Log-parabola β (curvature) |
| `log_nu_syn` | log10(`nu_syn`) | Treat 0 as NaN |
| `log_nuFnu_syn` | log10(`nuFnu_syn`) | Treat 0 as NaN |
| `log_he_epeak` | log10(`HE_EPeak`) | MeV, treat 0 as NaN |
| `log_compton_dom` | log10(`HE_nuFnuPeak` × 1.602e-6 / `nuFnu_syn`) | Convert MeV/cm²/s → erg/cm²/s |
| `log_gamma_lum` | log10(4π d_L² × `Energy_Flux100`) | d_L from cosmo.luminosity_distance(z) |
| `var_index` | `Variability_Index` | |
| `frac_var` | `Frac_Variability` | Treat ≤ 0 as NaN |

**Part 2 — 334 MOJAVE-matched sources, adds:**

| Feature | Notes |
|---------|-------|
| `beta_app` | `betaMax` from MOJAVE, in units of c |
| `e_beta_app` | `e_betaMax` |
| `gamma_min` | sqrt(1 + β_app²) — lower bound on Lorentz factor; flag betaMax=0 sources |

Save both as parquet.

### Build Part 1 dataframe (all 4LAC-DR3-h sources)

In [ ]:
# Build Part 2 dataframe (MOJAVE cross-matched sources)


In [ ]:
# Save both dataframes


## 6. Missingness Audit

Document missingness for every feature. This drives feature set decisions in all downstream steps.

**Tasks:**
- Print % missing per feature for Part 1 and Part 2
- Plot a missingness heatmap (a sample of 500 rows is sufficient)
- Identify co-missing features: `nu_syn`=0 and `SED_class`=missing are the same 777 sources
- **Decision:** for Part 1 clustering, run with two feature sets: (a) SED features only (no luminosity), (b) SED + luminosity (redshift-complete subsample only). Compare results.

In [ ]:
# Missingness audit — Part 1


In [ ]:
# Missingness audit — Part 2


## 7. Exploratory Data Analysis

Understand the structure of the data before applying any ML. Any surprise here should change the modeling approach.

### 7.1 Univariate distributions
Histograms / KDE per feature, colored by CLASS (BLL=blue, FSRQ=red, BCU=gray). Look for bimodality, skewness, outliers.

### 7.2 BLL vs. FSRQ separation
Pairplot: `log_nu_syn`, `log_compton_dom`, `redshift`. The blazar sequence predicts BLLs at higher synchrotron peak, lower Compton dominance, and generally lower redshift than FSRQs.

### 7.3 Blazar sequence
Scatter: `log_nu_syn` vs. `log_compton_dom`, colored by SED_class (LSP/ISP/HSP). This is the canonical blazar sequence diagram — does the data reproduce it?

### 7.4 Correlation matrix
Pearson correlation heatmap on the complete-case subset. Flag strongly correlated pairs for the modeling phase.

In [ ]:
# 7.1 Univariate distributions


In [ ]:
# 7.2 BLL vs FSRQ pairplot


In [ ]:
# 7.3 Blazar sequence: log_nu_syn vs log_compton_dom


In [ ]:
# 7.4 Correlation matrix


## 8. Dimensionality Reduction: PCA and UMAP

**Feature set:** complete-case sources on core SED features: `pl_index`, `log_nu_syn`, `log_nuFnu_syn`, `log_compton_dom`, `var_index`. StandardScaler before both methods.

### 8.1 PCA
- Scree plot (explained variance ratio)
- PC1 vs PC2 colored by: CLASS, SED_class, log_nu_syn
- Loadings: does PC1 correspond to the blazar sequence?

### 8.2 UMAP
- `n_neighbors=15`, `min_dist=0.1`, `random_state=42`
- Plot colored by CLASS and SED_class
- Key question: where do BCU sources land relative to confirmed BLL and FSRQ?
- Install if needed: `pip install umap-learn`

In [ ]:
# 8.1 PCA


In [ ]:
# 8.2 UMAP


## 9. Clustering: GMM and HDBSCAN

**Goal:** let the data reveal its own groupings without imposing BLL/FSRQ labels. Then compare to known classifications.

### 9.1 Gaussian Mixture Model
- Fit for k = 2, 3, 4, 5 on the same feature set as Section 8
- Select k using BIC and AIC
- Plot cluster assignments on the UMAP embedding
- Confusion matrix: GMM clusters vs. BLL/FSRQ/BCU labels

### 9.2 HDBSCAN
- Fit on UMAP embedding (2D), not raw features
- Start with `min_cluster_size=30`
- Inspect noise points (label = -1): are they predominantly BCU?
- Install if needed: `pip install hdbscan`

### 9.3 BCU probabilistic classification
- Using the GMM posterior probabilities, assign soft class labels to BCU sources
- Report: how many BCUs fall clearly in one class? How many are genuinely intermediate?
- This answers the secondary scientific question

In [ ]:
# 9.1 GMM


In [ ]:
# 9.2 HDBSCAN


In [ ]:
# 9.3 BCU probabilistic classification
